# Phase 2: Google Colab ML Environment Verification

This notebook verifies the Google Colab execution environment, checks GPU availability, mounts Google Drive, sets up persistent artifact directories, installs dependencies, and exports `environment_report.json` for all downstream training pipelines.

In [ ]:
# Step 1: Verify Python Version and Runtime Environment
import sys
import platform
import os

print("=== Python & System Info ===")
print(f"Python Version: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"System: {platform.system()} {platform.release()}")

# Informational environment check
if sys.version_info < (3, 10):
    raise RuntimeError(f"Python 3.10+ required, but found {sys.version_info.major}.{sys.version_info.minor}")
print("✓ Python runtime version validated.")

In [ ]:
# Step 2: GPU Detection & CUDA Verification (Fail Fast)
import torch

print("=== GPU & CUDA Verification ===")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if not cuda_available:
    raise RuntimeError(
        "\n" + "="*70 + "\n"
        "CRITICAL ERROR: No GPU detected in this Colab session!\n"
        "Model training requires a GPU runtime.\n"
        "Please switch the runtime accelerator:\n"
        "  1. In the Colab menu, click 'Runtime' -> 'Change runtime type'\n"
        "  2. Under 'Hardware accelerator', select 'T4 GPU' (or A100/V100)\n"
        "  3. Click 'Save' and re-run this notebook.\n"
        + "="*70
    )

gpu_count = torch.cuda.device_count()
gpu_name = torch.cuda.get_device_name(0)
cuda_version = torch.version.cuda
cudnn_version = torch.backends.cudnn.version()
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print(f"✓ GPU Detected: {gpu_name}")
print(f"✓ GPU Count: {gpu_count}")
print(f"✓ Total VRAM: {gpu_memory_gb:.2f} GB")
print(f"✓ CUDA Version: {cuda_version}")
print(f"✓ cuDNN Version: {cudnn_version}")

In [ ]:
# Step 3 & 4: Mount Google Drive and Setup Persistent Artifact Directories
from pathlib import Path

print("=== Google Drive & Artifact Directory Setup ===")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    drive_mounted = True
    print("✓ Google Drive mounted successfully at /content/drive")
except ImportError:
    print("Running outside Google Colab environment. Using local root fallback for path verification.")
    drive_mounted = False

# Define base project storage paths on Drive
BASE_PROJECT_DIR = Path("/content/drive/MyDrive/pd_voice_project")
CHECKPOINT_DIR = BASE_PROJECT_DIR / "checkpoints"
ARTIFACT_DIR = BASE_PROJECT_DIR / "artifacts"
EXPORT_DIR = BASE_PROJECT_DIR / "exported"

for directory in [BASE_PROJECT_DIR, CHECKPOINT_DIR, ARTIFACT_DIR, EXPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
    print(f"✓ Directory verified: {directory}")

In [ ]:
# Step 5: Install & Inspect Package Dependencies
import subprocess
import sys

print("=== Installing & Verifying Required Dependencies ===")
packages = [
    "torch",
    "torchaudio",
    "timm",
    "librosa",
    "soundfile",
    "praat-parselmouth",
    "opensmile",
    "faster-whisper",
    "shap",
    "captum",
    "scikit-learn",
    "pyyaml"
]

# Check requirements file if present, or install package list
req_file = Path("requirements-colab.txt")
if req_file.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)])
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)

import importlib.metadata
installed_versions = {}
pkg_module_map = {
    "torch": "torch",
    "torchaudio": "torchaudio",
    "timm": "timm",
    "librosa": "librosa",
    "soundfile": "soundfile",
    "praat-parselmouth": "praat-parselmouth",
    "opensmile": "opensmile",
    "faster-whisper": "faster-whisper",
    "shap": "shap",
    "captum": "captum",
    "scikit-learn": "scikit-learn",
    "pyyaml": "PyYAML"
}

for pkg, dist_name in pkg_module_map.items():
    try:
        ver = importlib.metadata.version(dist_name)
        installed_versions[pkg] = ver
        print(f"  - {pkg}: {ver}")
    except Exception as e:
        installed_versions[pkg] = f"Installed (version lookup error: {e})"
        print(f"  - {pkg}: verified")

print("✓ All required packages installed and verified.")

In [ ]:
# Step 6 & 7: Generate environment_report.json and Persist to Drive
import json
from datetime import datetime, timezone

report = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
    "total_vram_gb": round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if torch.cuda.is_available() else 0.0,
    "cuda_version": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "drive_mounted": drive_mounted,
    "storage_paths": {
        "base_dir": str(BASE_PROJECT_DIR),
        "checkpoints": str(CHECKPOINT_DIR),
        "artifacts": str(ARTIFACT_DIR),
        "exported": str(EXPORT_DIR)
    },
    "installed_packages": installed_versions
}

# Persist to Drive artifacts directory
report_path = ARTIFACT_DIR / "environment_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

# Also save a local backup copy in current working directory
with open("environment_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("=== Environment Report Successfully Created ===")
print(f"Artifact written to: {report_path}")
print("\n--- environment_report.json Contents ---")
print(json.dumps(report, indent=2))
print("\n✓ Environment check complete and validated.")